In [1]:
import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


In [2]:
import os
import numpy as np
import torch
import torchvision.models as models
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import RadarPointCloud
from pyquaternion import Quaternion
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.cluster import DBSCAN
torch.cuda.empty_cache()

In [3]:
DATA_ROOT = "/media/nas_mount/anwar2/experiment/dataset/nuscenes"
VERSION = "v1.0-trainval"
OUT_DIR = "/media/nas_mount/anwar2/experiment/dataset/nuscenes/nischay/bev_features"
os.makedirs(OUT_DIR, exist_ok=True)


X_RANGE = (-50, 50)
Y_RANGE = (-50, 50)
RES = 0.5
BEV_SIZE = int((X_RANGE[1] - X_RANGE[0]) / RES)

NUM_OBJECTS = 80

In [4]:
nusc = NuScenes(version=VERSION, dataroot=DATA_ROOT, verbose=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= IMAGE BACKBONE =================

# Load pretrained ResNet50
weights = ResNet50_Weights.DEFAULT
img_backbone = resnet50(weights=weights)

# Remove avgpool + fc → keep spatial feature map
img_backbone = torch.nn.Sequential(*list(img_backbone.children())[:-2])
img_backbone = img_backbone.to(device)
img_backbone.eval()

img_proj = torch.nn.Conv2d(
    in_channels=2048,
    out_channels=64,
    kernel_size=1,
    bias=False
)
img_proj = img_proj.to(device)
img_proj.eval()

# ================= IMAGE TRANSFORM =================

img_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/anwar/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 114MB/s]


In [5]:
def bev_coords(x, y):
    ix = ((x - X_RANGE[0]) / RES).astype(np.int32)
    iy = ((y - Y_RANGE[0]) / RES).astype(np.int32)
    return ix, iy


def radar_to_bev(radar_pc):
    bev = np.zeros((BEV_SIZE, BEV_SIZE, 5), dtype=np.float32)

    x = radar_pc.points[0]
    y = radar_pc.points[1]
    vx = radar_pc.points[8]
    vy = radar_pc.points[9]
    v = np.sqrt(vx ** 2 + vy ** 2)
    rcs = radar_pc.points[5]

    ix, iy = bev_coords(x, y)
    mask = (ix >= 0) & (ix < BEV_SIZE) & (iy >= 0) & (iy < BEV_SIZE)

    for i, j, vel, refl in zip(ix[mask], iy[mask], v[mask], rcs[mask]):
        bev[j, i, 0] += 1
        bev[j, i, 1] += vel
        bev[j, i, 2] = max(bev[j, i, 2], vel)
        bev[j, i, 3] += refl
        bev[j, i, 4] = 1

    bev[..., 1] /= (bev[..., 0] + 1e-6)
    bev[..., 3] /= (bev[..., 0] + 1e-6)

    return bev

In [6]:
def image_to_bev(image):
    """
    Weak but acceptable BEV proxy:
    collapse image features spatially into BEV grid
    """
    img = img_tf(image).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = img_backbone(img)
        feat = img_proj(feat)[0]   # (64, H, W)

    feat = feat.cpu().numpy()
    C, H, W = feat.shape

    bev = np.zeros((BEV_SIZE, BEV_SIZE, C), dtype=np.float32)
    bev[:H, :W, :] = feat.transpose(1, 2, 0)

    return bev


def fuse_bev(img_bev, radar_bev):
    return np.concatenate([img_bev, radar_bev], axis=-1)


def bev_to_objects(bev, topk=NUM_OBJECTS):
    scores = bev[..., -1]  # radar occupancy
    flat = scores.flatten()
    idx = np.argsort(flat)[::-1][:topk]

    objs = []
    for i in idx:
        y = i // BEV_SIZE
        x = i % BEV_SIZE
        objs.append(bev[y, x])

    return np.stack(objs)

# def bev_to_objects(radar_pc, bev, topk=NUM_OBJECTS):

#     # radar points in BEV plane
#     pts = radar_pc.points[:2].T   # (N,2) -> x,y

#     # cluster radar detections
#     clustering = DBSCAN(eps=1.5, min_samples=3).fit(pts)

#     labels = clustering.labels_

#     objs = []

#     for lab in set(labels):
#         if lab == -1:
#             continue

#         cluster = pts[labels == lab]

#         # object center
#         cx = cluster[:,0].mean()
#         cy = cluster[:,1].mean()

#         ix, iy = bev_coords(cx, cy)

#         if 0 <= ix < BEV_SIZE and 0 <= iy < BEV_SIZE:
#             objs.append(bev[iy, ix])

#     if len(objs) == 0:
#         return np.zeros((topk, bev.shape[-1]), dtype=np.float32)

#     objs = np.array(objs)

#     # pad / trim to fixed size
#     if len(objs) > topk:
#         objs = objs[:topk]
#     else:
#         pad = np.zeros((topk - len(objs), bev.shape[-1]))
#         objs = np.vstack([objs, pad])

#     return objs


In [7]:
for scene in tqdm(nusc.scene, desc="Scenes"):
    sample_token = scene["first_sample_token"]

    while sample_token:
        sample = nusc.get("sample", sample_token)

        # ---------- CAMERA ----------
        cam_sd = nusc.get("sample_data", sample["data"]["CAM_FRONT"])
        img_path = os.path.join(DATA_ROOT, cam_sd["filename"])
        image = Image.open(img_path).convert("RGB")

        cam_cs = nusc.get("calibrated_sensor", cam_sd["calibrated_sensor_token"])
        cam_pose = nusc.get("ego_pose", cam_sd["ego_pose_token"])

        # ---------- RADAR ----------
        radar_sd = nusc.get("sample_data", sample["data"]["RADAR_FRONT"])
        radar_pc = RadarPointCloud.from_file(
            os.path.join(DATA_ROOT, radar_sd["filename"])
        )

        radar_cs = nusc.get("calibrated_sensor", radar_sd["calibrated_sensor_token"])
        radar_pose = nusc.get("ego_pose", radar_sd["ego_pose_token"])

        # Radar → Ego
        radar_pc.rotate(Quaternion(radar_cs["rotation"]).rotation_matrix)
        radar_pc.translate(np.array(radar_cs["translation"]))

        # Ego → Global
        radar_pc.rotate(Quaternion(radar_pose["rotation"]).rotation_matrix)
        radar_pc.translate(np.array(radar_pose["translation"]))

        # Global → Ego (camera)
        radar_pc.translate(-np.array(cam_pose["translation"]))
        radar_pc.rotate(Quaternion(cam_pose["rotation"]).rotation_matrix.T)

        # ---------- BEV ----------
        radar_bev = radar_to_bev(radar_pc)
        img_bev = image_to_bev(image)

        bev = fuse_bev(img_bev, radar_bev)
        obj_feat = bev_to_objects(bev)
        # obj_feat = bev_to_objects(radar_pc, bev)

        # ---------- SAVE ----------
        np.save(
            os.path.join(OUT_DIR, f"{sample_token}.npy"),
            obj_feat.astype(np.float32)
        )

        sample_token = sample["next"]

print("✅ Image + Radar BEV feature extraction complete.")


Scenes: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 850/850 [25:36<00:00,  1.81s/it]

✅ Image + Radar BEV feature extraction complete.
